In [1]:
import pandas as pd
import numpy as np

# # Load the dataset
# data = pd.read_csv('creditcard')
import os;
file_path = os.path.join('/kaggle/input/ccdf-dataset/', 'creditcard.csv')

# Create the DataFrame
data = pd.read_csv(file_path)

# View the first few rows
data.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [2]:
from sklearn.model_selection import train_test_split

# Assuming 'df' is your DataFrame and 'Class' is your target column
# we take 25% as the subset
sample, _ = train_test_split(
    data, 
    test_size=0.75,       # We toss 75%, keeping 25%
    stratify=data['Class'], # This is the magic line for stratification
    random_state=42       # Ensures you get the same sample every time you run it
)

print(f"Original shape: {data.shape}")
print(f"Sample shape: {sample.shape}")
print(f"Sample Class Distribution:\n{sample['Class'].value_counts(normalize=True)}")

Original shape: (284807, 31)
Sample shape: (71201, 31)
Sample Class Distribution:
Class
0    0.998272
1    0.001728
Name: proportion, dtype: float64


In [3]:
from sklearn.model_selection import train_test_split

# Separate features (X) and target (y)
X = sample.drop('Class', axis=1)
y = sample['Class']

# Perform a stratified train-test split to ensure both sets have the same fraud ratio (0.17%)
# Using 80% for training and 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler,StandardScaler

In [5]:
## Handle Imbalance (SMOTE Example for Training Data)

from imblearn.over_sampling import SMOTE

# Initialize SMOTE
smote = SMOTE(sampling_strategy='minority', random_state=42)

# Apply SMOTE ONLY to the training data
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"Original Training Size: {len(y_train)}")
print(f"Original Training value counts: {y_train.value_counts()}\n")

print(f"SMOTE Training Size: {len(y_train_smote)}")
print(f"SMOTE Training value counts: {y_train_smote.value_counts()}\n")

print(f"SMOTE Training Fraud Rate: {y_train_smote.mean():.4f}")

Original Training Size: 49840
Original Training value counts: Class
0    49754
1       86
Name: count, dtype: int64

SMOTE Training Size: 99508
SMOTE Training value counts: Class
0    49754
1    49754
Name: count, dtype: int64

SMOTE Training Fraud Rate: 0.5000


In [6]:
np.random.seed(0)
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ("classifier", LogisticRegression(random_state=0))
])

param_grid_lr_sm = param_grid_lr_sm = [
    {
        "classifier": [LogisticRegression(random_state=0)],
        "classifier__penalty": ['l1','l2'],
        "classifier__C": [0.1,0.01,0.001],
        "classifier__solver": ['liblinear'],
        "classifier__class_weight": [None],
        "classifier__max_iter": [1000,2000]
    }]

# Define Scorer (Good practice for imbalanced data like fraud detection)
# Using AUPRC (Average Precision Score) as suggested in the dataset context. It focuses only on the minority class (fraud).
# AUPRC gives a true picture of performance without being skewed by the large number of easy-to-classify non-fraud cases
scorer = make_scorer(average_precision_score)

grid_lr_sm = GridSearchCV(
    estimator=pipe_lr,
    scoring='average_precision',
    param_grid=param_grid_lr_sm,
    cv=5,
    n_jobs=-1
)

grid_lr_sm.fit(X_train_smote, y_train_smote)
print("Best score: {:.2f}".format(grid_lr_sm.best_score_))
print("Test set score: {:.2f}".format(grid_lr_sm.score(X_test, y_test)))
print("Best parameters: {}".format(grid_lr_sm.best_params_))

Best score: 1.00
Test set score: 0.74
Best parameters: {'classifier': LogisticRegression(random_state=0), 'classifier__C': 0.1, 'classifier__class_weight': None, 'classifier__max_iter': 1000, 'classifier__penalty': 'l1', 'classifier__solver': 'liblinear'}


In [7]:
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score, classification_report
pred = grid_lr_sm.best_estimator_.predict(X_test)
proba = grid_lr_sm.best_estimator_.predict_proba(X_test)[:,1]

print("Accuracy:", accuracy_score(y_test, pred))
print("ROC AUC:", roc_auc_score(y_test, proba))
print("AUPRC:", average_precision_score(y_test, proba))
print(classification_report(y_test, pred))

Accuracy: 0.9936800711577173
ROC AUC: 0.97853579522122
AUPRC: 0.7218214747689099
              precision    recall  f1-score   support

           0       1.00      0.99      1.00     21324
           1       0.20      0.86      0.32        37

    accuracy                           0.99     21361
   macro avg       0.60      0.93      0.66     21361
weighted avg       1.00      0.99      1.00     21361



In [8]:
pipe_rf = Pipeline([
    ('scaler', StandardScaler()),
    ("classifier",RandomForestClassifier())
    ])

from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "classifier__n_estimators": [100, 200, 300],
    "classifier__max_features": ['sqrt', 'log2'],
    "classifier__max_depth": [None, 10, 20],
    "classifier__min_samples_split": [5,10,20],
    "classifier__criterion": ['entropy'],
    "classifier__n_jobs": [-1],
    "classifier__class_weight": [None]
}

grid_rf_sm = RandomizedSearchCV(
    estimator=pipe_rf,
    param_distributions=param_dist,
    n_iter=50,        # 🔥 instead of 108
    cv=5,             # reduce folds
    n_jobs=-1,
)


# Fit grid search
grid_rf_sm.fit(X_train_smote, y_train_smote)

# Return all parameters and components of the pipeline as a dictionary
grid_rf_sm.best_estimator_.get_params()

# View best model
grid_rf_sm.best_estimator_.get_params()["classifier"]

# Predict target vector
grid_rf_sm.predict(X_test)

print("Best accuracy: {:.2f}".format(grid_rf_sm.best_score_))
print("Test set score: {:.2f}".format(grid_rf_sm.score(X_test, y_test)))
print("Best parameters: {}".format(grid_rf_sm.best_params_))

results_rf = grid_rf_sm.cv_results_

Best accuracy: 1.00
Test set score: 1.00
Best parameters: {'classifier__n_jobs': -1, 'classifier__n_estimators': 100, 'classifier__min_samples_split': 5, 'classifier__max_features': 'log2', 'classifier__max_depth': 20, 'classifier__criterion': 'entropy', 'classifier__class_weight': None}


In [9]:
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score, classification_report
pred = grid_rf_sm.best_estimator_.predict(X_test)
proba = grid_rf_sm.best_estimator_.predict_proba(X_test)[:,1]

print("Accuracy:", accuracy_score(y_test, pred))
print("ROC AUC:", roc_auc_score(y_test, proba))
print("AUPRC:", average_precision_score(y_test, proba))
print(classification_report(y_test, pred))

Accuracy: 0.9994850428350732
ROC AUC: 0.9256154719716903
AUPRC: 0.8054209357305344
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     21324
           1       0.88      0.81      0.85        37

    accuracy                           1.00     21361
   macro avg       0.94      0.91      0.92     21361
weighted avg       1.00      1.00      1.00     21361



In [10]:
!pip install xgboost

In [11]:
from collections import Counter

counter = Counter(y_train)
scale_pos_weight = counter[0] / counter[1]

pipe_xgb = Pipeline([
    ('scaler', StandardScaler()),
    ("classifier",xgb.XGBClassifier())])

In [12]:
search_space_xgb_sm = [
               {"classifier": [xgb.XGBClassifier()],
                 "classifier__n_estimators": [10,100,200],
                 "classifier__max_depth": [None,6,10],
                 "classifier__learning_rate": [0.01, 0.05, 0.1],
                 "classifier__n_jobs": [-1],
                 "classifier__gamma": [0.5,0,1],
                 "classifier__scale_pos_weight": [None]
                }]

# Create grid search
grid_xgb_sm = GridSearchCV(
    estimator=pipe_xgb,
    param_grid=search_space_xgb_sm,
    scoring='average_precision',
    cv=5,
    n_jobs=-1
)
# Fit grid search
grid_xgb_sm.fit(X_train_smote, y_train_smote)

# Return all parameters and components of the pipeline as a dictionary
grid_xgb_sm.best_estimator_.get_params()

# View best model
grid_xgb_sm.best_estimator_.get_params()["classifier"]

# Predict target vector
grid_xgb_sm.predict(X_test)

print("Best accuracy: {:.2f}".format(grid_xgb_sm.best_score_))
print("Test set score: {:.2f}".format(grid_xgb_sm.score(X_test, y_test)))
print("Best parameters: {}".format(grid_xgb_sm.best_params_))

results_xgb = grid_xgb_sm.cv_results_

Best accuracy: 1.00
Test set score: 0.81
Best parameters: {'classifier': XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...), 'classifier__gamma': 0, 'classifier__learning_rate': 0.1, 'classifier__max_depth': 10, 'classifier__n_estimators': 200, 'classifier__n_jobs': -1, 'classifier__scale_

In [13]:
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score, classification_report
pred = grid_xgb_sm.best_estimator_.predict(X_test)
proba = grid_xgb_sm.best_estimator_.predict_proba(X_test)[:,1]

print("Accuracy:", accuracy_score(y_test, pred))
print("ROC AUC:", roc_auc_score(y_test, proba))
print("AUPRC:", average_precision_score(y_test, proba))
print(classification_report(y_test, pred))

Accuracy: 0.9992509713964702
ROC AUC: 0.9783760969748589
AUPRC: 0.8149445431730029
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     21324
           1       0.77      0.81      0.79        37

    accuracy                           1.00     21361
   macro avg       0.88      0.91      0.89     21361
weighted avg       1.00      1.00      1.00     21361

